In [7]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

print(f"현재 사용 중인 연산 디바이스: {device}")

현재 사용 중인 연산 디바이스: cpu


c:\Users\boomd\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\torch\cuda\__init__.py:188: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:88.)
  return torch._C._cuda_getDeviceCount() > 0


In [8]:
import torch.nn as nn
import torch.optim as optim

class AdvancedFashionClassifier(nn.Module):

    # 사용할 층(layer)를 정의
    def __init__(self):
        super().__init__()

        # (28, 28) -> 특성은 1차원으로 변경(784,)
        # nn.Flatter() -> 2차원 텐서 -> 1차원 텐서
        self.flatten = nn.Flatten()

        # 은닉층 (784 -> 128)
        self.hidden_layer = nn.Linear(784, 64)

        # 활성화 함수(ReLU)
        self.relu = nn.ReLU()

        # Dropout - 0 ~ 1, 0.1 ~ 0.5
        self.dropout = nn.Dropout(p=0.2)

        # 출력층 128 -> 10
        self.output_layer = nn.Linear(64, 10)

    # 순전파(Feed Forward)
    def forward(self, x):
        out = self.flatten(x) # 28 X 27 -> 784 # 입력층 처리
        out = self.hidden_layer(out) # 784 -> 128 # 은닉층 처리
        out = self.relu(out) # 활성화 함수 통과(선형적 -> 비선형적)
        out = self.dropout(out) # 훈련시에만 필요, 과대적합 방지 (128,)
        out = self.output_layer(out) # 출력층 128 -> 10

        return out


In [ ]:
# Fashion MNIST 28 X 28 형태 (0~255) -> 특성 데이터로 쓰려면 1차원으로 변환
# 60,000 -> 훈련 데이터는 50,000장, 검증 10,000장

# 훈련 데이터 셋
raw_train_data = datasets.FashionMNIST(
    root="data", train=True, download=True, transform=ToTensor()
)

# 테스트 데이터셋
test_data = datasets.FashionMNIST(
    root="data", train=False, download=True, transform=ToTensor()
)

In [10]:
# 검증 데이터셋 분리
train_size = 50000
val_size = 10000

train_data, val_data = random_split(
    raw_train_data, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("훈련 세트:", len(train_data))
print("검증 세트:", len(val_data))
print("테스트 세트:", len(test_data))

훈련 세트: 50000
검증 세트: 10000
테스트 세트: 10000


In [11]:
# 데이터 세트별 DataLoader 생성
# batch_size -> 에포크에서 전 데이터셋을 32개 배치로 분할
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [12]:
train_data[0]

(tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1529, 0.2353, 0.2000,
           0.2235, 0.1961, 0.2157, 0.2078, 0.1961, 0.1922, 0.1647, 0.1725,
           0.1804, 0.1843, 0.2353, 0.0431],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0078, 0.0000, 0.5216, 0.6196, 0.5922,
           0.7529, 0.7725, 0.8118, 0.7882, 

In [13]:
model = AdvancedFashionClassifier().to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 가능한 총 모델 파라미터 갯수: {total_params:,}개")

학습 가능한 총 모델 파라미터 갯수: 50,890개
